# 📝 博客发布助手

这个 Notebook 帮你完成两件事：

1. **内置编辑器** —— 输入框里直接写 Markdown，点按钮一键发布
2. **Markdown 速查** —— 完整语法对照表，照着写就行

> 技术栈：Python（标准库 + ipywidgets），博客是 **Hexo + GitHub Pages**，目录在 `C:\Users\Operator\Desktop\blog`


## 0. 激活环境（重要！）

本 Notebook 用 **博客专属环境** `blog`，运行前请确认：

- 右上角 Kernel 选择 **`Blog (Python 3.13)`**（即 conda 的 `blog` 环境）
- 如果没看到，点击内核选择器 → 选「Blog (Python 3.13)」

```bash
# 在终端确认 blog 环境
conda activate blog
python --version   # 应显示 Python 3.13.x
```

> 为什么用 `blog` 环境？因为它和博客一体：既有 **Python 3.13**（跑本 Notebook 的转换/发布逻辑），又有 **Node.js 26**（跑 Hexo 构建），发布流程全在一个环境里搞定，不污染你的 `STU` 学习环境。
> 本 Notebook 只用标准库 + ipywidgets，不需要额外安装任何包。


In [ ]:
# 1. 导入所需库 + 配置博客路径
# 只用标准库，任何 Python 3 都能跑

import os            # 处理文件路径
import datetime      # 生成文章日期
import subprocess    # 执行 git 命令（提交推送）

# 博客根目录（如果博客换位置了，改这一行即可）
BLOG_DIR  = r"C:\Users\Operator\Desktop\blog"
POSTS_DIR = os.path.join(BLOG_DIR, "source", "_posts")   # 文章存放目录

print("✅ 博客路径：", BLOG_DIR)
print("✅ 文章目录：", POSTS_DIR, "存在：", os.path.isdir(POSTS_DIR))


## 3. Markdown 格式速查（自己写，别怕）

博客文章就是一个 `.md` 文件，格式规则就下面这些，10 分钟学会 👇

### 📐 标题
```markdown
# 一级标题（文章大标题，一般不手动写）
## 二级标题（小节标题，最常用）
### 三级标题（小节下的小点）
```

### 📝 段落与换行
```markdown
段落之间用空行隔开，会自动分段。

这是第二段。
```
> ⚠️ 同一段内直接回车只是软换行，**想分段必须加空行**。

### 🖊 加粗 / 斜体 / 行内代码
```markdown
**加粗文字**
*斜体文字*
`行内代码`
```

### 📋 列表
```markdown
- 无序列表项
- 第二项

1. 有序列表项
2. 第二项
```

### 🔗 链接与图片
```markdown
[文字链接](https://example.com)
![图片说明](/images/图.jpg)   # 图片放博客的 source/images/ 目录
```

### 💻 代码块（三个反引号包裹）
```markdown
​```python
print("hello")
​```
```

### 📊 表格
```markdown
| 列1 | 列2 |
|-----|-----|
| A   | B   |
```

### 💬 引用与分割线
```markdown
> 引用别人的话

---
```

### ⚠️ 文章头部（front matter）别删
每篇文章开头必须有这段"元信息"，保存时会自动生成（也可手动改）：

```yaml
---
title: 文章标题
date: 2026-08-07 12:00:00
tags:
  - 技术
categories:
  - 技术
---
```

💡 写完正文后，运行下面「保存 + 发布」单元格即可上线。


In [ ]:
# 5. 保存为博客文章 + 一键发布
# 把你写好的 Markdown 存进 source/_posts/，再 git 提交推送 → GitHub Actions 自动部署

def save_post(title: str, content: str) -> str:
    """
    保存为一篇 Hexo 文章（自动加 front matter 模板）
    - title:   文章标题
    - content: Markdown 正文（你自己写的）
    - 返回：保存的文件路径
    """
    today = datetime.date.today().isoformat()
    # front matter 是 Hexo 认识文章"元信息"的头部（标题/日期/分类/标签）
    front = (
        "---\n"
        f"title: {title}\n"
        f"date: {today} 12:00:00\n"
        "tags:\n"
        "  - 技术\n"
        "categories:\n"
        "  - 技术\n"
        "---\n\n"
    )
    safe = title.replace("/", "-").replace("\\", "-").replace(":", "：")
    path = os.path.join(POSTS_DIR, f"{safe}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(front + content)
    print(f"✅ 文章已保存：{path}")
    return path


def git_publish(title: str):
    """
    提交并推送博客改动（GitHub Actions 会自动构建部署）
    """
    print("🔄 git add / commit / push …")
    # 依次执行三条 git 命令
    cmds = [
        ['git', '-C', BLOG_DIR, 'add', '.'],
        ['git', '-C', BLOG_DIR, 'commit', '-m', f'新文章：{title}'],
        ['git', '-C', BLOG_DIR, 'push'],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
        out = (r.stdout or "").strip()
        err = (r.stderr or "").strip()
        if out: print(out)
        if r.returncode != 0 and "error" in err.lower(): print(err)
    print("🎉 已推送！1-2 分钟后访问 https://gutianshuo.github.io")


# ── 用法：先保存，再发布 ──────────────
# 1. 自己写一段 Markdown（参考上面的「Markdown 速查」）
# my_md = "## 我的第一节\n\n这是正文，支持 **加粗**、- 列表 等语法…"
#    （也可以直接用 VS Code 在 source/_posts/ 里写 .md）
# 2. 保存（自动加 front matter）→ 推送上线
# save_post("我的新文章", my_md)
# git_publish("我的新文章")


In [2]:
# 6. 内置编辑器：直接在 Notebook 里写文章
# ⭐ 本单元格是「自包含」的：保存 + 发布函数都内置在这里，
#    你只需要运行【这一个】单元格，就能拥有完整编辑器，不用先跑前面的代码！

# ── ① 内置依赖（标准库，任何 Python 3 都能跑）────────────────
import os            # 处理文件路径
import datetime      # 生成文章日期
import subprocess    # 执行 git 命令（提交推送）

# 博客根目录（如果博客换位置了，改这一行即可）
BLOG_DIR  = r"C:\Users\Operator\Desktop\blog"
POSTS_DIR = os.path.join(BLOG_DIR, "source", "_posts")   # 文章存放目录

# ── ② 保存 + 发布函数（原本在 cell 5，已合并进来）────────────
def save_post(title: str, content: str) -> str:
    """
    保存为一篇 Hexo 文章（自动加 front matter 模板）
    - title:   文章标题
    - content: Markdown 正文（你自己写的）
    - 返回：保存的文件路径
    """
    today = datetime.date.today().isoformat()
    # front matter 是 Hexo 认识文章"元信息"的头部（标题/日期/分类/标签）
    front = (
        "---\n"
        f"title: {title}\n"
        f"date: {today} 12:00:00\n"
        "tags:\n"
        "  - 技术\n"
        "categories:\n"
        "  - 技术\n"
        "---\n\n"
    )
    safe = title.replace("/", "-").replace("\\", "-").replace(":", "：")
    path = os.path.join(POSTS_DIR, f"{safe}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(front + content)
    print(f"✅ 文章已保存：{path}")
    return path


def git_publish(title: str):
    """
    提交并推送博客改动（GitHub Actions 会自动构建部署）
    """
    print("🔄 git add / commit / push …")
    cmds = [
        ['git', '-C', BLOG_DIR, 'add', '.'],
        ['git', '-C', BLOG_DIR, 'commit', '-m', f'新文章：{title}'],
        ['git', '-C', BLOG_DIR, 'push'],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
        out = (r.stdout or "").strip()
        err = (r.stderr or "").strip()
        if out: print(out)
        if r.returncode != 0 and "error" in err.lower(): print(err)
    print("🎉 已推送！1-2 分钟后访问 https://gutianshuo.github.io")


# ── ③ 输入框界面（ipywidgets 交互组件）──────────────────────
import ipywidgets as widgets

# 标题输入框
title_box = widgets.Text(
    description="标题：",
    placeholder="输入文章标题…",
    layout=widgets.Layout(width="90%"),
)

# Markdown 正文编辑器（多行文本框）
editor = widgets.Textarea(
    description="正文：",
    placeholder="在这里写 Markdown（语法见上面的速查）…",
    layout=widgets.Layout(width="90%", height="320px"),
)

# 两个按钮：只保存草稿 / 保存并发布
btn_save = widgets.Button(description="💾 保存草稿", button_style="primary")
btn_pub  = widgets.Button(description="🚀 保存并发布", button_style="success")
out = widgets.Output()


def get_title():
    """读取标题，为空则提示"""
    t = title_box.value.strip()
    if not t:
        with out:
            out.clear_output()
            print("❌ 请先填写标题")
    return t


def on_save(_):
    """点击「保存草稿」：只生成文章文件，不推送"""
    with out:
        out.clear_output()
        t = get_title()
        if t:
            save_post(t, editor.value)


def on_publish(_):
    """点击「保存并发布」：保存 + git 推送上线"""
    with out:
        out.clear_output()
        t = get_title()
        if t:
            save_post(t, editor.value)
            git_publish(t)


btn_save.on_click(on_save)
btn_pub.on_click(on_publish)

# 显示编辑器界面
display(widgets.VBox([title_box, editor, widgets.HBox([btn_save, btn_pub]), out]))

## 9. 完整使用流程总结

### 日常发布一篇文章
① 运行「导入配置」cell，选好 blog 内核
② 运行「内置编辑器」cell，直接在输入框里写 Markdown（语法看速查）
③ 点「💾 保存草稿」可先保存，点「🚀 保存并发布」直接上线
④ 1-2 分钟后访问 https://gutianshuo.github.io 刷新查看

### 小贴士
- **写 Markdown 没把握**？把内容粘贴给 Copilot 说「帮我转成 Markdown」，AI 帮你排版
- **评论区**：博客已装 Giscus（评论存 GitHub Discussions，无审查）
- 想用 VS Code 写也行：直接在 `source/_posts/` 新建 `.md` 文件

### 本 Notebook 用到的关键知识点
| 知识点 | 在哪 |
|---|---|
| conda 环境与 kernel | cell 0 |
| Markdown 语法 | cell 3（速查） |
| Hexo front matter 与 git 自动部署 | cell 5 |
| ipywidgets 输入框交互 | cell 6 |
